# CFR+ 30-Claim LR Schedule Screen

Runs four 60-minute neural CFR+ schedules on the 30-claim spec. Every 10 minutes it saves a playable average-policy snapshot and trains a fresh action-conditioned approximate best responder for 5 minutes, with MC evaluation once per responder minute.

No resumable `.pt` trainer checkpoints are written; artifacts are policy snapshots and JSONL logs only.


In [ ]:
from __future__ import annotations

from datetime import datetime, timezone
import gc
import json
import math
from pathlib import Path
import re
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / 'liars_poker').is_dir():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from liars_poker.algo.br_fitted_return_action_conditioned import ActionConditionedFittedReturnBRTrainer
from liars_poker.algo.deep_cfr_plus import DeepCFRPlusTrainer
from liars_poker.core import GameSpec
from liars_poker.env import rules_for_spec
from liars_poker.eval.approx_br import evaluate_approximate_br
from liars_poker.serialization import load_policy, save_policy

assert torch.cuda.is_available(), 'This notebook is intended for the RTX 3090 CUDA VM.'
device = torch.device('cuda')
torch.set_float32_matmul_precision('high')

spec = GameSpec(
    ranks=5,
    suits=4,
    hand_size=3,
    claim_kinds=('RankHigh', 'Pair', 'TwoPair', 'Trips', 'Quads'),
    suit_symmetry=True,
)
assert len(rules_for_spec(spec).claims) == 30

RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S')
RUN_ROOT = REPO_ROOT / 'artifacts' / 'cfr_plus_30_claim_lr_schedule_screen' / f'{spec.to_short_str()}___{RUN_ID}'
RUN_ROOT.mkdir(parents=True, exist_ok=True)

print('repo:', REPO_ROOT)
print('run root:', RUN_ROOT)
print('torch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0))
print('free / total GPU GiB:', tuple(round(x / 2**30, 2) for x in torch.cuda.mem_get_info()))
print('spec:', spec.to_short_str(), 'claims:', len(rules_for_spec(spec).claims))


In [ ]:
# Runtime controls. Default SEEDS=[17] matches the expected ~4 x (60 + 30) minute runtime.
# Add more seeds later if you want replication: e.g. SEEDS = [17, 27, 37].
SEEDS = [17]
TRAIN_MINUTES = 60.0
SNAPSHOT_EVERY_MINUTES = 10.0
BR_MINUTES = 5.0
BR_EVALUATE_EVERY_MINUTES = 1.0
PRINT_EVERY_MINUTES = 5.0

TRAVERSALS_PER_PLAYER = 1024
VALIDATION_EVERY_ITERATIONS = 50
VALIDATION_RECORDS = 4096

BASE_TRAINER_KWARGS = {
    'regret_hidden_sizes': (2048, 2048),
    'strategy_hidden_sizes': (512, 512),
    'device': device,
    'regret_buffer_capacity': 2_000_000,
    'strategy_buffer_capacity': 1_000_000,
    'learning_rate': 1e-3,
    'batch_size': 4096,
    'regret_train_steps': 24,
    'strategy_train_steps': 6,
    'strategy_weighting': 'linear',
    'regret_positive_weight': 0.5,
    'validation_fraction': 0.001,
    'validation_buffer_capacity': 20_000,
    'traversal_backend': 'gpu_native',
    'traversal_batch_size': 1024,
    # Full expansion for the 30-claim spec. Keep sampling off to isolate LR.
    'traverser_action_sample_count': None,
    'traverser_action_sample_fraction': None,
    'traverser_action_sample_schedule': None,
    'traverser_action_baseline': 'none',
    'traverser_action_sample_mode': 'random',
    'traversal_streaming': False,
    'traversal_live_row_budget': None,
    'traverser_action_chunk_size': None,
    'traversal_record_flush_size': 131_072,
    'device_replay': True,
    'fused_optimizer': True,
    'amp_dtype': None,
    'compile_models': False,
}

BR_TRAINER_KWARGS = {
    'state_hidden_sizes': (512, 512),
    'action_hidden_sizes': (128, 128),
    'embedding_dim': 256,
    'replay_capacity': 1_000_000,
    'batch_size': 4096,
    'lr': 1e-3,
    'train_steps': 100,
    'warmup_transitions': 20_000,
    'epsilon_start': 1.0,
    'epsilon_end': 0.05,
    'epsilon_decay_decisions': 500_000,
    'rollouts_per_action': 1,
    'fused_optimizer': True,
    'device': device,
}


In [ ]:
def smooth_cosine_lr(elapsed_min: float, total_min: float) -> float:
    start = 1e-3
    end = 1e-4
    progress = min(1.0, max(0.0, elapsed_min / total_min))
    return end + 0.5 * (start - end) * (1.0 + math.cos(math.pi * progress))


def very_aggressive_lr(elapsed_min: float, total_min: float) -> float:
    if elapsed_min < 20.0:
        return 1e-3
    if elapsed_min < 40.0:
        return 1e-4
    return 3e-5


def aggressive_moderate_lr(elapsed_min: float, total_min: float) -> float:
    if elapsed_min < 40.0:
        return 1e-3
    return 1e-4


def normal_late_lr(elapsed_min: float, total_min: float) -> float:
    # With a 60-minute horizon this is effectively constant 1e-3; it is the late-drop baseline.
    if elapsed_min < 60.0:
        return 1e-3
    return 1e-4


SCHEDULES = [
    {
        'name': 'smooth_cosine_1e3_to_1e4',
        'label': 'smooth cosine 1e-3?1e-4',
        'lr_fn': smooth_cosine_lr,
    },
    {
        'name': 'very_aggressive_20_40',
        'label': 'very aggressive 20/40: 1e-3?1e-4?3e-5',
        'lr_fn': very_aggressive_lr,
    },
    {
        'name': 'aggressive_moderate_40',
        'label': 'aggressive moderate 40m: 1e-3?1e-4',
        'lr_fn': aggressive_moderate_lr,
    },
    {
        'name': 'normal_late_drop_constant',
        'label': 'normal late drop: constant 1e-3 over 60m',
        'lr_fn': normal_late_lr,
    },
]

pd.DataFrame([{k: v for k, v in s.items() if k != 'lr_fn'} for s in SCHEDULES])


In [ ]:
def json_default(obj):
    if isinstance(obj, Path):
        return str(obj)
    if isinstance(obj, torch.device):
        return str(obj)
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, torch.Tensor):
        return obj.detach().cpu().tolist()
    if isinstance(obj, tuple):
        return list(obj)
    raise TypeError(type(obj).__name__)


def append_jsonl(path: Path, row: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('a', encoding='utf-8') as handle:
        handle.write(json.dumps(row, default=json_default, sort_keys=True) + '\n')


def read_jsonl(path: Path) -> list[dict]:
    if not path.exists():
        return []
    return [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]


def set_optimizer_lr(trainer: DeepCFRPlusTrainer, lr: float) -> None:
    for optimizer in [*trainer.regret_optimizers, *trainer.strategy_optimizers]:
        for group in optimizer.param_groups:
            group['lr'] = float(lr)


def slugify(text: str) -> str:
    return re.sub(r'[^a-z0-9]+', '_', text.lower()).strip('_')


def mean_or_nan(value) -> float:
    if value is None:
        return float('nan')
    if isinstance(value, (int, float)):
        return float(value)
    values = list(value)
    return float(np.mean(values)) if values else float('nan')


def gpu_mem_gib() -> tuple[float, float]:
    free, total = torch.cuda.mem_get_info()
    return round(free / 2**30, 2), round(total / 2**30, 2)


manifest = {
    'run_type': 'cfr_plus_30_claim_lr_schedule_screen',
    'spec': spec.to_json(),
    'seeds': SEEDS,
    'train_minutes': TRAIN_MINUTES,
    'snapshot_every_minutes': SNAPSHOT_EVERY_MINUTES,
    'br_minutes': BR_MINUTES,
    'traversals_per_player': TRAVERSALS_PER_PLAYER,
    'base_trainer_kwargs': {k: (str(v) if k == 'device' else v) for k, v in BASE_TRAINER_KWARGS.items()},
    'schedules': [{k: v for k, v in schedule.items() if k != 'lr_fn'} for schedule in SCHEDULES],
}
(RUN_ROOT / 'manifest.json').write_text(json.dumps(manifest, indent=2, default=json_default), encoding='utf-8')


In [ ]:
def run_snapshot_br(policy_dir: Path, *, schedule: dict, seed: int, target_min: float) -> pd.DataFrame:
    label = f"{schedule['label']} seed {seed} snapshot {target_min:04.0f}m"
    br_dir = RUN_ROOT / schedule['name'] / f'seed_{seed}' / 'br' / f'{int(target_min):04d}m'
    summary_path = br_dir / 'summary.json'
    if summary_path.exists():
        rows = read_jsonl(br_dir / 'evaluations.jsonl')
        return pd.DataFrame(rows)

    policy, loaded_spec = load_policy(str(policy_dir))
    if loaded_spec != spec:
        raise ValueError(f'Spec mismatch for {policy_dir}: {loaded_spec}')

    kwargs = dict(BR_TRAINER_KWARGS)
    kwargs['seed'] = int(seed + 10_000 + target_min)
    trainer = ActionConditionedFittedReturnBRTrainer(spec, policy, **kwargs)

    br_dir.mkdir(parents=True, exist_ok=True)
    measured_s = 0.0
    next_eval_s = 60.0 * BR_EVALUATE_EVERY_MINUTES
    print(f"  [BR] {label}")
    while measured_s < 60.0 * BR_MINUTES:
        start = time.perf_counter()
        record = trainer.run_iteration(episodes_per_role=4096, rollout_batch_size=1024)
        iter_s = time.perf_counter() - start
        measured_s += iter_s
        append_jsonl(br_dir / 'training.jsonl', {
            'schedule': schedule['label'],
            'schedule_name': schedule['name'],
            'seed': int(seed),
            'snapshot_target_min': float(target_min),
            'iteration': int(trainer.iteration),
            'measured_responder_s': float(measured_s),
            'measured_responder_min': float(measured_s / 60.0),
            'iteration_s': float(iter_s),
            'record': record,
        })

        if measured_s >= next_eval_s or measured_s >= 60.0 * BR_MINUTES:
            metrics = evaluate_approximate_br(
                trainer,
                episodes_per_role=200_000,
                rollout_batch_size=8192,
            )
            row = {
                'schedule': schedule['label'],
                'schedule_name': schedule['name'],
                'seed': int(seed),
                'snapshot_target_min': float(target_min),
                'policy_dir': str(policy_dir),
                'br_iteration': int(trainer.iteration),
                'responder_training_min': float(measured_s / 60.0),
                **metrics,
            }
            append_jsonl(br_dir / 'evaluations.jsonl', row)
            append_jsonl(RUN_ROOT / 'br_evaluations.jsonl', row)
            print(
                f"    BR={measured_s/60:.1f}m estimate={metrics['exploitability_estimate']:.5f} "
                f"LCB={metrics['exploitability_lower_bound']:.5f}"
            )
            while next_eval_s <= measured_s:
                next_eval_s += 60.0 * BR_EVALUATE_EVERY_MINUTES

    final_rows = read_jsonl(br_dir / 'evaluations.jsonl')
    summary = final_rows[-1] if final_rows else {}
    summary_path.write_text(json.dumps(summary, indent=2, default=json_default), encoding='utf-8')

    del trainer, policy
    gc.collect()
    torch.cuda.empty_cache()
    return pd.DataFrame(final_rows)


In [ ]:
def run_schedule(schedule: dict, seed: int) -> dict:
    schedule_dir = RUN_ROOT / schedule['name'] / f'seed_{seed}'
    summary_path = schedule_dir / 'summary.json'
    if summary_path.exists():
        print('already complete:', schedule['label'], 'seed', seed)
        return json.loads(summary_path.read_text(encoding='utf-8'))

    schedule_dir.mkdir(parents=True, exist_ok=True)
    kwargs = dict(BASE_TRAINER_KWARGS)
    kwargs['seed'] = int(seed)
    trainer = DeepCFRPlusTrainer(spec, **kwargs)

    measured_s = 0.0
    budget_s = 60.0 * TRAIN_MINUTES
    snapshot_targets_s = [60.0 * minute for minute in np.arange(SNAPSHOT_EVERY_MINUTES, TRAIN_MINUTES + 0.1, SNAPSHOT_EVERY_MINUTES)]
    next_snapshot_index = 0
    next_print_s = 0.0
    start_wall = time.perf_counter()

    print(f"\n=== {schedule['label']} | seed {seed} ===")
    print('free / total GPU GiB:', gpu_mem_gib())
    while measured_s < budget_s:
        lr = float(schedule['lr_fn'](measured_s / 60.0, TRAIN_MINUTES))
        set_optimizer_lr(trainer, lr)

        iter_start = time.perf_counter()
        record = trainer.run_iteration(traversals_per_player=TRAVERSALS_PER_PLAYER)
        iter_s = time.perf_counter() - iter_start
        measured_s += iter_s

        timing = record.get('timing', {})
        row = {
            'schedule': schedule['label'],
            'schedule_name': schedule['name'],
            'seed': int(seed),
            'iteration': int(trainer.iteration),
            'measured_training_s': float(measured_s),
            'measured_training_min': float(measured_s / 60.0),
            'wall_s': float(time.perf_counter() - start_wall),
            'iteration_s': float(iter_s),
            'learning_rate': lr,
            'traversals_per_player': int(TRAVERSALS_PER_PLAYER),
            'traversal_s': float(timing.get('traversal_s', np.nan)),
            'regret_fit_s': float(timing.get('regret_training_s', np.nan)),
            'strategy_fit_s': float(timing.get('strategy_training_s', np.nan)),
            'regret_loss': mean_or_nan(record.get('regret_loss')),
            'strategy_loss': mean_or_nan(record.get('strategy_loss')),
            'new_regret_records': int(sum(record.get('new_regret_records') or [])),
            'new_strategy_records': int(sum(record.get('new_strategy_records') or [])),
        }
        if trainer.iteration % VALIDATION_EVERY_ITERATIONS == 0:
            row['validation'] = trainer.validation_metrics(max_records=VALIDATION_RECORDS)
        append_jsonl(schedule_dir / 'training.jsonl', row)

        while next_snapshot_index < len(snapshot_targets_s) and measured_s >= snapshot_targets_s[next_snapshot_index]:
            target_min = float(snapshot_targets_s[next_snapshot_index] / 60.0)
            snapshot_dir = schedule_dir / 'snapshots' / f'{int(target_min):04d}m'
            policy_dir = snapshot_dir / 'policy'
            save_policy(trainer.average_policy(), str(policy_dir))
            append_jsonl(schedule_dir / 'snapshots.jsonl', {
                'schedule': schedule['label'],
                'schedule_name': schedule['name'],
                'seed': int(seed),
                'snapshot_target_min': target_min,
                'snapshot_training_min': float(measured_s / 60.0),
                'snapshot_iteration': int(trainer.iteration),
                'policy_dir': str(policy_dir),
            })
            print(f"[snapshot] target={target_min:.0f}m actual={measured_s/60:.1f}m iter={trainer.iteration}")
            run_snapshot_br(policy_dir, schedule=schedule, seed=seed, target_min=target_min)
            gc.collect()
            torch.cuda.empty_cache()
            next_snapshot_index += 1

        if measured_s >= next_print_s:
            print(
                f"train={measured_s/60:.1f}m iter={trainer.iteration} lr={lr:.2e} "
                f"iter_s={iter_s:.3f} trav={row['traversal_s']:.3f}s "
                f"fit={row['regret_fit_s']:.3f}/{row['strategy_fit_s']:.3f}s"
            )
            next_print_s += 60.0 * PRINT_EVERY_MINUTES

    final_policy_dir = schedule_dir / 'final_policy'
    save_policy(trainer.average_policy(), str(final_policy_dir))
    training_rows = read_jsonl(schedule_dir / 'training.jsonl')
    summary = {
        'schedule': schedule['label'],
        'schedule_name': schedule['name'],
        'seed': int(seed),
        'status': 'complete',
        'iterations': int(trainer.iteration),
        'measured_training_min': float(measured_s / 60.0),
        'final_policy_dir': str(final_policy_dir),
        'mean_traversal_s': float(np.mean([r['traversal_s'] for r in training_rows])),
        'mean_regret_fit_s': float(np.mean([r['regret_fit_s'] for r in training_rows])),
        'mean_strategy_fit_s': float(np.mean([r['strategy_fit_s'] for r in training_rows])),
    }
    summary_path.write_text(json.dumps(summary, indent=2, default=json_default), encoding='utf-8')
    append_jsonl(RUN_ROOT / 'schedule_summaries.jsonl', summary)

    del trainer
    gc.collect()
    torch.cuda.empty_cache()
    return summary


In [ ]:
# Long run cell. Expected default runtime is roughly 4 x (60m training + 30m BR) plus MC eval overhead.
all_summaries = []
for seed in SEEDS:
    for schedule in SCHEDULES:
        all_summaries.append(run_schedule(schedule, seed))

summary_df = pd.DataFrame(all_summaries)
display(summary_df)


In [ ]:
# Plot and summarize all available results. Safe to rerun while or after the long run.
training_rows = []
for schedule in SCHEDULES:
    for seed in SEEDS:
        training_rows.extend(read_jsonl(RUN_ROOT / schedule['name'] / f'seed_{seed}' / 'training.jsonl'))
train_df = pd.DataFrame(training_rows)
br_df = pd.DataFrame(read_jsonl(RUN_ROOT / 'br_evaluations.jsonl'))

if not br_df.empty:
    final_br = (
        br_df.sort_values('responder_training_min')
        .groupby(['schedule', 'seed', 'snapshot_target_min'])
        .tail(1)
        .sort_values(['snapshot_target_min', 'exploitability_estimate'])
    )
    display(
        final_br[[
            'schedule', 'seed', 'snapshot_target_min', 'responder_training_min',
            'p_first', 'p_second', 'exploitability_estimate', 'exploitability_lower_bound',
        ]]
        .style.format({
            'snapshot_target_min': '{:.0f}',
            'responder_training_min': '{:.2f}',
            'p_first': '{:.5f}',
            'p_second': '{:.5f}',
            'exploitability_estimate': '{:.5f}',
            'exploitability_lower_bound': '{:.5f}',
        })
    )

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

if not br_df.empty:
    final_br = br_df.sort_values('responder_training_min').groupby(['schedule', 'seed', 'snapshot_target_min']).tail(1)
    for (schedule_label, seed), group in final_br.groupby(['schedule', 'seed']):
        group = group.sort_values('snapshot_target_min')
        label = f'{schedule_label} | seed {seed}'
        axes[0, 0].plot(group['snapshot_target_min'], group['exploitability_estimate'], marker='o', label=label)
        axes[0, 1].plot(group['snapshot_target_min'], group['exploitability_lower_bound'], marker='o', label=label)

    for (schedule_label, seed, snapshot_min), group in br_df.groupby(['schedule', 'seed', 'snapshot_target_min']):
        if snapshot_min == max(br_df['snapshot_target_min']):
            axes[1, 0].plot(group['responder_training_min'], group['exploitability_estimate'], marker='o', label=f'{schedule_label} | seed {seed}')

if not train_df.empty:
    for (schedule_label, seed), group in train_df.groupby(['schedule', 'seed']):
        group = group.sort_values('measured_training_min')
        axes[1, 1].plot(group['measured_training_min'], group['learning_rate'], label=f'{schedule_label} | seed {seed}')

axes[0, 0].set_title('Approximate exploitability by CFR+ training time')
axes[0, 1].set_title('Conservative lower bound by CFR+ training time')
axes[1, 0].set_title('Final-snapshot BR compute curves')
axes[1, 1].set_title('Learning-rate schedules actually used')

axes[0, 0].set_xlabel('CFR+ training minutes')
axes[0, 1].set_xlabel('CFR+ training minutes')
axes[1, 0].set_xlabel('Responder training minutes')
axes[1, 1].set_xlabel('CFR+ training minutes')
axes[0, 0].set_ylabel('Discovered exploitability')
axes[0, 1].set_ylabel('Conservative lower bound')
axes[1, 0].set_ylabel('Discovered exploitability')
axes[1, 1].set_ylabel('Learning rate')
axes[1, 1].set_yscale('log')

for ax in axes.ravel():
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=7)

plt.tight_layout()
plt.show()

if not train_df.empty:
    timing = (
        train_df.groupby(['schedule', 'seed'])[['traversal_s', 'regret_fit_s', 'strategy_fit_s']]
        .mean()
        .reset_index()
    )
    display(timing)
